In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as pl
import warnings

from xgboost.testing import root_mean_square

warnings.filterwarnings("ignore")

ModuleNotFoundError: No module named 'pytest'

In [ ]:
data = pd.read_csv('stud.csv')
data.info()

In [ ]:
data['gender']= data['gender'].map({'male':0, 'female':1})

In [ ]:
data['parental_level_of_education'] = data['parental_level_of_education'].replace(
    'some high school',
    'high school'
)

In [ ]:
mapping = {
    'high school': 0,
    'some college': 1,
    "associate's degree": 2,
    "bachelor's degree": 3,
    "master's degree": 4
}

data['parental_level_of_education'] = data['parental_level_of_education'].map(mapping)

In [ ]:
data['test_preparation_course']=data['test_preparation_course'].map({'none':0,'completed':1})

In [ ]:
data.duplicated().sum()

In [ ]:
data[data.duplicated()]

In [ ]:
print(f"Unique values : \n{data.nunique()}")
print(f"Unique values sum : \n{data.nunique().sum()}")

In [ ]:
data.describe()

In [ ]:
data = data.drop(columns=['race_ethnicity','lunch'])

In [ ]:
pl.figure()
sns.pairplot(data, hue='gender' ,kind='reg',diag_kind='hist')

In [ ]:
Q1 = data.quantile(.25)
Q3 = data.quantile(.75)
IQR = Q3 - Q1
Upper = Q3+1.5*IQR
Low = Q1-1.5*IQR
is_outlier = (data < Low) | (data > Upper)
clean_data = data[~is_outlier.any(axis=1)]

In [ ]:
pl.figure()
sns.pairplot(clean_data, hue='gender' ,kind='reg',diag_kind='hist')

In [ ]:
clean_data=clean_data.drop(columns=['parental_level_of_education','test_preparation_course'])
clean_data.info()

In [ ]:
clean_data['total_score'] = clean_data['math_score'] + clean_data['reading_score'] + clean_data['writing_score']
clean_data['average_score'] = clean_data['total_score']/3

In [ ]:
clean_data.info()

In [ ]:
pl.figure()
sns.histplot(data=clean_data,x=clean_data['average_score'] ,bins=50,hue='gender',kde=True)

In [ ]:
features = ['reading_score','writing_score']

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor,AdaBoostRegressor
from sklearn.svm import SVR
from sklearn. linear_model import LinearRegression, Ridge,Lasso
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
import warnings

In [ ]:
x = data.drop(columns=['math_score'])
y = data['math_score']

In [ ]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)

In [ ]:
def evaluate_model(true, predicted):
    mae = mean_absolute_error(true, predicted)
    mse = mean_squared_error(true, predicted)
    rmse = np.sqrt(mse)
    r2 = r2_score(true, predicted)

    return mae, mse, rmse, r2

In [ ]:
models = {
    "Linear Regression" : LinearRegression(),
    "Ridge Regression" : Ridge(),
    "Lasso Regression" : Lasso(),
    "Decision Tree Regression" : DecisionTreeRegressor(),
    "Random Forest Regression" : RandomForestRegressor(),
    "AdaBoost Regression" : AdaBoostRegressor(),
    "CatBoost Regression" : CatBoostRegressor(verbose=False),
    "XGBoost Regression" : XGBRegressor(),
    "KNN Regression" : KNeighborsRegressor(),
}
model_List = []
r2_list = []
for i in range(len(list(models))):

    model = list(models.values())[i]

    model.fit(x_train, y_train)

    y_train_pred = model.predict(x_train)
    y_test_pred = model.predict(x_test)

    # Evaluate model
    model_train_mae, model_train_mse, model_train_rmse, model_train_r2 = evaluate_model(y_train, y_train_pred)

    model_test_mae, model_test_mse, model_test_rmse, model_test_r2 = evaluate_model(y_test, y_test_pred)

    print(list(models.keys())[i])

    model_List.append(list(models.keys())[i])
    r2_list.append(model_test_r2)
    print("Model performance for training")
    print(f"RMSE:{model_train_rmse:.3f}")
    print(f"MSE:{model_train_mse:.3f}")
    print(f"MAE:{model_train_mae:.3f}")
    print(f"R2:{model_train_r2:.3f}")
    print("Model performance for testing")
    print(f"RMSE:{model_test_rmse:.3f}")
    print(f"MSE:{model_test_mse:.3f}")
    print(f"MAE:{model_test_mae:.3f}")
    print(f"R2:{model_test_r2:.3f}")

In [ ]:
pd.DataFrame(list(zip(model_List,r2_list)),columns=['Model','R2']).sort_values(by='R2',ascending=False)